# Notebook 06 — VodRec-Transformer (PyTorch from scratch)

**Projeto:** VOD-IA — PUC-Campinas

**Objetivo:** construir e treinar **um Transformer decoder-only do zero**, em PyTorch puro, para a tarefa de **next-item prediction** em sequências de visualização.

Não usamos `sklearn`, `implicit`, OpenAI, nem `transformers` da Hugging Face para a arquitetura. **Tudo é código próprio**: embeddings, multi-head attention, positional encoding, blocos transformer, loop de treino.

**Por que é um LLM:** o modelo aprende `P(c_{t+1} | c_1, ..., c_t)` — é exatamente um Language Model causal, com a única diferença de que o vocabulário são `content_id`s do catálogo VOD em vez de palavras.

**Recomendamos rodar no Colab com GPU** (Runtime → Change runtime type → T4).

## 1. Setup

In [ ]:
!pip install -q torch pandas numpy pyarrow tqdm matplotlib

In [ ]:
import math
import json
import os
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Preparação dos dados

Esperamos um `interactions.parquet` com colunas `user_id`, `content_id`, `started_at`, `rating_implicit`.
Se não tiver, gera dataset sintético (mesma lógica do notebook 01).

In [ ]:
DATA_DIR = Path('data')
MODELS_DIR = Path('models/vodrec')
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

INTERACTIONS_PATH = DATA_DIR / 'interactions.parquet'
CONTENTS_PATH     = DATA_DIR / 'contents.parquet'

In [ ]:
if not INTERACTIONS_PATH.exists():
    print('Gerando dataset sintético...')
    from faker import Faker  # noqa
    !pip install -q faker
    from faker import Faker
    import random
    fake = Faker('pt_BR'); Faker.seed(42); random.seed(42)

    GENRES = ['Acao','Aventura','Comedia','Drama','FiccaoCientifica','Terror',
              'Suspense','Romance','Documentario','Animacao','Fantasia','Crime',
              'Misterio','Historico','Musical']
    N_USERS, N_CONTENTS = 2000, 1500

    contents = []
    for cid in range(1, N_CONTENTS+1):
        n_g = random.choice([1,2,2,3])
        contents.append({
            'content_id': cid,
            'title': f'Titulo_{cid}',
            'genres': random.sample(GENRES, n_g),
            'duration_sec': random.choice([1800,2700,3600,5400,7200]),
        })
    contents_df = pd.DataFrame(contents)

    interactions = []
    for uid in range(1, N_USERS+1):
        favs = random.sample(GENRES, random.choice([1,2,3]))
        seen = set()
        n_int = max(3, int(np.random.normal(40, 15)))
        for t in range(n_int):
            if random.random() < 0.7:
                pool = contents_df[contents_df['genres'].apply(lambda g: any(x in favs for x in g))]['content_id']
            else:
                pool = contents_df['content_id']
            unseen = [c for c in pool.tolist() if c not in seen]
            if not unseen: break
            cid = random.choice(unseen); seen.add(cid)
            cdur = int(contents_df.loc[contents_df.content_id==cid,'duration_sec'].iloc[0])
            content_genres = contents_df.loc[contents_df.content_id==cid,'genres'].iloc[0]
            is_fav = any(g in favs for g in content_genres)
            completion = float(np.clip(np.random.beta(8 if is_fav else 2, 2 if is_fav else 4),0,1))
            interactions.append({
                'user_id': uid,
                'content_id': cid,
                'started_at': pd.Timestamp.now() - pd.Timedelta(days=n_int-t),
                'completion': completion,
                'rating_implicit': float(np.clip(0.6*completion + 0.1*(completion>0.9),0,1)),
            })
    interactions_df = pd.DataFrame(interactions)
    interactions_df.to_parquet(INTERACTIONS_PATH, index=False)
    contents_df.to_parquet(CONTENTS_PATH, index=False)
    print(f'Gerados {len(interactions_df)} interacoes, {len(contents_df)} conteudos, {N_USERS} usuarios.')
else:
    print(f'Carregando dataset existente de {INTERACTIONS_PATH}')
    interactions_df = pd.read_parquet(INTERACTIONS_PATH)
    contents_df = pd.read_parquet(CONTENTS_PATH)

print(f'Interacoes: {len(interactions_df)}')
print(f'Usuarios:   {interactions_df.user_id.nunique()}')
print(f'Conteudos:  {interactions_df.content_id.nunique()}')

## 3. Construção do vocabulário

Mapeia `content_id` → `token_id`:
- Token 0 → `<pad>` (padding)
- Token 1 → `<bos>` (begin-of-sequence, útil para cold start)
- Tokens 2..V+1 → conteúdos do catálogo

In [ ]:
SPECIAL_TOKENS = {'<pad>': 0, '<bos>': 1}

all_content_ids = sorted(interactions_df['content_id'].unique().tolist())
content_to_token = {**SPECIAL_TOKENS,
                    **{cid: i + len(SPECIAL_TOKENS) for i, cid in enumerate(all_content_ids)}}
token_to_content = {v: k for k, v in content_to_token.items()}
VOCAB_SIZE = len(content_to_token)
print(f'Tamanho do vocabulario: {VOCAB_SIZE} ({VOCAB_SIZE - len(SPECIAL_TOKENS)} conteudos + {len(SPECIAL_TOKENS)} especiais)')

# Salva o vocab
vocab_path = MODELS_DIR / 'vocab.json'
with open(vocab_path, 'w') as f:
    json.dump({
        'special_tokens': SPECIAL_TOKENS,
        'content_to_token': {str(k): v for k, v in content_to_token.items()},
    }, f, indent=2)
print(f'Vocab salvo em {vocab_path}')

## 4. Construção das sequências por usuário

In [ ]:
MAX_SEQ_LEN = 128
MIN_SEQ_LEN = 3    # sequencias menores nao entram no treino
POSITIVE_THRESHOLD = 0.4

# Filtra interacoes positivas e ordena por tempo
pos = interactions_df[interactions_df['rating_implicit'] >= POSITIVE_THRESHOLD].copy()
pos = pos.sort_values(['user_id', 'started_at'])

# Constrói sequências (lista de token_ids por usuário)
user_sequences = (
    pos.groupby('user_id')['content_id']
       .apply(lambda s: [content_to_token[c] for c in s.tolist()])
       .to_dict()
)
user_sequences = {u: s for u, s in user_sequences.items() if len(s) >= MIN_SEQ_LEN}

lens = [len(s) for s in user_sequences.values()]
print(f'Usuarios com sequencia valida: {len(user_sequences)}')
print(f'Tamanho seq: mean={np.mean(lens):.1f}, median={np.median(lens):.0f}, p95={np.percentile(lens,95):.0f}, max={max(lens)}')

In [ ]:
# Split temporal: ultimos 20% das interacoes de cada usuario viram teste
train_sequences = {}
test_targets = {}    # ultimo item de cada usuario para HR@k
test_full = {}       # sequencia completa de teste

for uid, seq in user_sequences.items():
    n_test = max(1, int(len(seq) * 0.2))
    train_sequences[uid] = seq[:-n_test]
    test_targets[uid] = seq[-1]
    test_full[uid] = seq[-n_test:]

print(f'Treino: {len(train_sequences)} sequencias')
print(f'Teste:  {len(test_targets)} targets')

## 5. Dataset PyTorch com sliding window

Para sequências mais longas que `MAX_SEQ_LEN`, criamos múltiplas janelas (data augmentation).

In [ ]:
class NextItemDataset(Dataset):
    def __init__(self, sequences, max_seq_len, stride=32, pad_id=0, bos_id=1):
        self.max_seq_len = max_seq_len
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.examples = []
        for uid, seq in sequences.items():
            seq = [bos_id] + seq  # prepend <bos>
            if len(seq) <= max_seq_len + 1:
                self.examples.append(seq)
            else:
                # sliding window: cada janela tem max_seq_len+1
                for start in range(0, len(seq) - max_seq_len, stride):
                    self.examples.append(seq[start:start + max_seq_len + 1])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        seq = self.examples[idx]
        # pad a esquerda ate max_seq_len+1
        if len(seq) < self.max_seq_len + 1:
            pad_len = self.max_seq_len + 1 - len(seq)
            seq = [self.pad_id] * pad_len + seq
        seq = torch.tensor(seq, dtype=torch.long)
        return seq[:-1], seq[1:]

train_ds = NextItemDataset(train_sequences, MAX_SEQ_LEN, stride=32)
print(f'Exemplos de treino: {len(train_ds)}')

x, y = train_ds[0]
print(f'Input shape:  {x.shape}, exemplo: {x[-10:].tolist()}')
print(f'Target shape: {y.shape}, exemplo: {y[-10:].tolist()}')

## 6. Arquitetura do VodRec-Transformer

Tudo escrito do zero: MultiHeadAttention, FeedForward, TransformerBlock, modelo completo.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention com causal masking."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, T, dh)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, h, T, T)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask == 0, float('-inf'))
        if key_padding_mask is not None:
            # key_padding_mask: (B, T) com True onde é padding
            scores = scores.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(1), float('-inf')
            )
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = attn @ v  # (B, h, T, dh)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Bloco estilo GPT: pre-LayerNorm."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        x = x + self.dropout(self.attn(self.ln1(x), attn_mask, key_padding_mask))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x


class VodRecTransformer(nn.Module):
    """Decoder-only Transformer para next-item prediction sobre catalogo VOD.
    Tied embeddings: item_emb e o head de saida compartilham pesos.
    """
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4,
                 max_seq_len=128, dropout=0.1, pad_id=0):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.pad_id = pad_id

        self.item_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb  = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        # Causal mask pré-computado
        self.register_buffer('causal_mask',
            torch.tril(torch.ones(max_seq_len, max_seq_len)).bool())

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, seq):
        # seq: (B, T) — token ids
        B, T = seq.shape
        assert T <= self.max_seq_len, f'seq len {T} > max {self.max_seq_len}'

        positions = torch.arange(T, device=seq.device).unsqueeze(0).expand(B, T)
        x = self.item_emb(seq) + self.pos_emb(positions)
        x = self.drop(x)

        causal = self.causal_mask[:T, :T]
        key_padding_mask = (seq == self.pad_id)

        for block in self.blocks:
            x = block(x, causal, key_padding_mask)
        x = self.ln_f(x)

        # Tied embeddings: logits = x @ item_emb.weight.T
        logits = x @ self.item_emb.weight.T  # (B, T, V)
        return logits

    def num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Sanity-check da arquitetura
test_model = VodRecTransformer(vocab_size=VOCAB_SIZE, d_model=128, n_heads=4,
                                n_layers=4, max_seq_len=MAX_SEQ_LEN).to(device)
print(f'Parametros treinaveis: {test_model.num_params():,}')

sample = torch.randint(0, VOCAB_SIZE, (2, 32)).to(device)
out = test_model(sample)
print(f'Input shape:  {sample.shape}')
print(f'Output shape: {out.shape}  (deve ser B=2, T=32, V={VOCAB_SIZE})')
del test_model

## 7. Loop de treino

In [ ]:
CONFIG = {
    'vocab_size':  VOCAB_SIZE,
    'd_model':     128,
    'n_heads':     4,
    'n_layers':    4,
    'max_seq_len': MAX_SEQ_LEN,
    'dropout':     0.1,
    'pad_id':      0,
    'batch_size':  256,
    'lr':          3e-4,
    'weight_decay':0.01,
    'epochs':      15,
    'warmup_steps':500,
    'label_smoothing': 0.1,
}

model = VodRecTransformer(
    vocab_size=CONFIG['vocab_size'], d_model=CONFIG['d_model'],
    n_heads=CONFIG['n_heads'], n_layers=CONFIG['n_layers'],
    max_seq_len=CONFIG['max_seq_len'], dropout=CONFIG['dropout'],
    pad_id=CONFIG['pad_id'],
).to(device)

print(f'Modelo: {model.num_params():,} parametros (~{model.num_params()/1e6:.1f}M)')

loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                    num_workers=2, pin_memory=(device.type=='cuda'))
total_steps = len(loader) * CONFIG['epochs']

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'],
                               weight_decay=CONFIG['weight_decay'])

def lr_lambda(step):
    if step < CONFIG['warmup_steps']:
        return step / max(1, CONFIG['warmup_steps'])
    progress = (step - CONFIG['warmup_steps']) / max(1, total_steps - CONFIG['warmup_steps'])
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
def evaluate_hr_ndcg(model, sequences, targets, k=10, max_users=500):
    """Hit Rate@k e NDCG@k sobre amostra de usuarios."""
    model.eval()
    hr_hits, ndcg_sum, n = 0, 0.0, 0
    uids = list(targets.keys())[:max_users]
    with torch.no_grad():
        for uid in uids:
            history = sequences.get(uid)
            target = targets[uid]
            if not history:
                continue
            seq = [1] + history[-(MAX_SEQ_LEN-1):]  # <bos> + ultimos N
            if len(seq) < MAX_SEQ_LEN:
                seq = [0]*(MAX_SEQ_LEN - len(seq)) + seq
            x = torch.tensor([seq[-MAX_SEQ_LEN:]], dtype=torch.long, device=device)
            logits = model(x)[0, -1]  # (V,)
            # Mascara seen + pad/bos
            logits[0] = -1e9; logits[1] = -1e9
            for tok in history:
                logits[tok] = -1e9
            topk = torch.topk(logits, k=k).indices.tolist()
            if target in topk:
                hr_hits += 1
                rank = topk.index(target)
                ndcg_sum += 1.0 / math.log2(rank + 2)
            n += 1
    return hr_hits / max(1, n), ndcg_sum / max(1, n)

In [ ]:
history = {'train_loss': [], 'val_hr10': [], 'val_ndcg10': [], 'lr': []}
best_hr = 0.0
BEST_PATH = MODELS_DIR / 'best_model.pt'

for epoch in range(1, CONFIG['epochs']+1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(loader, desc=f'Epoch {epoch}/{CONFIG["epochs"]}', leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1),
            ignore_index=CONFIG['pad_id'],
            label_smoothing=CONFIG['label_smoothing'],
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.3f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    avg_loss = epoch_loss / len(loader)
    hr10, ndcg10 = evaluate_hr_ndcg(model, train_sequences, test_targets, k=10)

    history['train_loss'].append(avg_loss)
    history['val_hr10'].append(hr10)
    history['val_ndcg10'].append(ndcg10)
    history['lr'].append(scheduler.get_last_lr()[0])

    print(f'  Epoch {epoch}: loss={avg_loss:.4f}  HR@10={hr10:.4f}  NDCG@10={ndcg10:.4f}')

    if hr10 > best_hr:
        best_hr = hr10
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': CONFIG,
            'epoch': epoch,
            'hr10': hr10,
            'ndcg10': ndcg10,
        }, BEST_PATH)
        print(f'  -> Salvou checkpoint (best HR@10 = {hr10:.4f})')

print(f'\nTreinamento concluido. Melhor HR@10: {best_hr:.4f}')

## 8. Curvas de aprendizado

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(history['train_loss'], 'o-', color='steelblue')
axes[0].set_title('Train loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_hr10'], 'o-', color='coral', label='HR@10')
axes[1].axhline(0.70, color='red', linestyle='--', label='RFIA01')
axes[1].plot(history['val_ndcg10'], 'o-', color='teal', label='NDCG@10')
axes[1].set_title('Validation metrics'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(history['lr'], color='purple')
axes[2].set_title('Learning rate'); axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Recarrega o melhor checkpoint e exemplifica uma recomendação

In [ ]:
ckpt = torch.load(BEST_PATH, map_location=device)
model = VodRecTransformer(
    vocab_size=ckpt['config']['vocab_size'], d_model=ckpt['config']['d_model'],
    n_heads=ckpt['config']['n_heads'], n_layers=ckpt['config']['n_layers'],
    max_seq_len=ckpt['config']['max_seq_len'], dropout=0.0, pad_id=0,
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Carregou checkpoint da epoca {ckpt["epoch"]} com HR@10={ckpt["hr10"]:.4f}')

In [ ]:
def recommend_for_user(model, history_tokens, k=10, exclude_seen=True, temperature=1.0):
    """Recebe lista de token_ids (mais antigo primeiro) e retorna [(token_id, prob), ...]."""
    model.eval()
    seq = [1] + list(history_tokens[-(MAX_SEQ_LEN-1):])
    if len(seq) < MAX_SEQ_LEN:
        seq = [0]*(MAX_SEQ_LEN - len(seq)) + seq
    seq = seq[-MAX_SEQ_LEN:]
    x = torch.tensor([seq], dtype=torch.long, device=device)
    with torch.no_grad():
        logits = model(x)[0, -1] / temperature
    logits[0] = -1e9; logits[1] = -1e9
    if exclude_seen:
        for tok in history_tokens:
            logits[tok] = -1e9
    probs = F.softmax(logits, dim=-1)
    topv, topi = torch.topk(probs, k=k)
    return list(zip(topi.tolist(), topv.tolist()))

# Exemplo
sample_uid = next(iter(train_sequences))
hist = train_sequences[sample_uid]
print(f'Usuario {sample_uid} — historico ({len(hist)} itens, ultimos 10):')
for t in hist[-10:]:
    cid = token_to_content[t]
    title = contents_df.loc[contents_df.content_id==cid, 'title'].iloc[0] if cid in contents_df.content_id.values else '???'
    print(f'  token={t}  content_id={cid}  title="{title}"')

print(f'\nTop-10 recomendados:')
for token_id, prob in recommend_for_user(model, hist, k=10):
    cid = token_to_content[token_id]
    title = contents_df.loc[contents_df.content_id==cid, 'title'].iloc[0] if cid in contents_df.content_id.values else '???'
    genres = contents_df.loc[contents_df.content_id==cid, 'genres'].iloc[0] if cid in contents_df.content_id.values else []
    print(f'  [{prob:.4f}] content_id={cid}  title="{title}"  genres={genres}')

## 10. Avaliação completa (HR@10, NDCG@10, MAP@10)

In [ ]:
def evaluate_full(model, train_seq, test_full_seq, k=10):
    """Para cada usuario, mede se cada item do teste aparece no top-k."""
    model.eval()
    hits_at_k = []
    ndcg = []
    mrr = []
    with torch.no_grad():
        for uid, hist in tqdm(train_seq.items(), desc='Eval'):
            test_items = test_full_seq.get(uid)
            if not hist or not test_items:
                continue
            recs = [t for t, _ in recommend_for_user(model, hist, k=k)]
            hit = any(t in test_items for t in recs)
            hits_at_k.append(1 if hit else 0)
            # ndcg considerando o primeiro item de teste como ground truth principal
            gt = test_items[0]
            if gt in recs:
                rank = recs.index(gt)
                ndcg.append(1.0 / math.log2(rank + 2))
                mrr.append(1.0 / (rank + 1))
            else:
                ndcg.append(0); mrr.append(0)
    return {
        'n_users': len(hits_at_k),
        f'HR@{k}': float(np.mean(hits_at_k)),
        f'NDCG@{k}': float(np.mean(ndcg)),
        f'MRR@{k}': float(np.mean(mrr)),
    }

metrics = evaluate_full(model, train_sequences, test_full, k=10)
print('\n=== Avaliacao final VodRec-Transformer ===')
for k, v in metrics.items():
    print(f'  {k:12s}: {v if isinstance(v,int) else round(v,4)}')
print(f'\nRFIA01 (HR@10 >= 0.70): {"OK" if metrics["HR@10"] >= 0.70 else "AINDA NAO — tunar hyperparams"}')

## 11. Benchmark de latência (RFIA02)

In [ ]:
import time
lats_gpu = []
for uid in list(train_sequences.keys())[:200]:
    hist = train_sequences[uid]
    t0 = time.perf_counter()
    recommend_for_user(model, hist, k=20)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    lats_gpu.append((time.perf_counter() - t0) * 1000)
lats_gpu = np.array(lats_gpu)
print(f'Latencia ({device.type.upper()}):')
print(f'  P50  = {np.percentile(lats_gpu, 50):.2f} ms')
print(f'  P95  = {np.percentile(lats_gpu, 95):.2f} ms')
print(f'  P99  = {np.percentile(lats_gpu, 99):.2f} ms')
print(f'  Max  = {lats_gpu.max():.2f} ms')

# Inferencia em CPU (mais realista para producao)
model_cpu = VodRecTransformer(**{k:v for k,v in ckpt['config'].items() if k in
                                  ['vocab_size','d_model','n_heads','n_layers','max_seq_len','dropout','pad_id']})
model_cpu.load_state_dict(ckpt['model_state_dict'])
model_cpu.eval()

def recommend_cpu(history_tokens, k=20):
    seq = [1] + list(history_tokens[-(MAX_SEQ_LEN-1):])
    if len(seq) < MAX_SEQ_LEN:
        seq = [0]*(MAX_SEQ_LEN - len(seq)) + seq
    seq = seq[-MAX_SEQ_LEN:]
    x = torch.tensor([seq], dtype=torch.long)
    with torch.no_grad():
        logits = model_cpu(x)[0, -1]
    logits[0] = -1e9; logits[1] = -1e9
    for t in history_tokens: logits[t] = -1e9
    topi = torch.topk(F.softmax(logits, dim=-1), k=k).indices.tolist()
    return topi

lats_cpu = []
for uid in list(train_sequences.keys())[:50]:
    t0 = time.perf_counter()
    recommend_cpu(train_sequences[uid], k=20)
    lats_cpu.append((time.perf_counter() - t0) * 1000)
lats_cpu = np.array(lats_cpu)
print(f'\nLatencia CPU (producao mais realista):')
print(f'  P50  = {np.percentile(lats_cpu, 50):.2f} ms')
print(f'  P95  = {np.percentile(lats_cpu, 95):.2f} ms')
print(f'  RFIA02 (<2000ms): {"OK" if np.percentile(lats_cpu, 95) < 2000 else "FALHOU"}')

## 12. Export final

In [ ]:
FINAL_MODEL_PATH = MODELS_DIR / 'model.pt'
CONFIG_PATH      = MODELS_DIR / 'config.json'
VERSION_PATH     = MODELS_DIR / 'VERSION.txt'
METRICS_PATH     = MODELS_DIR / 'metrics.json'

torch.save({
    'model_state_dict': model.state_dict(),
    'config': ckpt['config'],
}, FINAL_MODEL_PATH)

with open(CONFIG_PATH, 'w') as f:
    json.dump(ckpt['config'], f, indent=2)

with open(VERSION_PATH, 'w') as f:
    f.write('vodrec-v1.0.0')

with open(METRICS_PATH, 'w') as f:
    json.dump({
        'metrics': metrics,
        'latency_gpu_ms': {
            'p50': float(np.percentile(lats_gpu, 50)),
            'p95': float(np.percentile(lats_gpu, 95)),
        },
        'latency_cpu_ms': {
            'p50': float(np.percentile(lats_cpu, 50)),
            'p95': float(np.percentile(lats_cpu, 95)),
        },
        'vocab_size': VOCAB_SIZE,
        'parameters': model.num_params(),
    }, f, indent=2)

print('Salvo:')
for f in sorted(MODELS_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

## 13. Próximo passo

- Notebook **07_vodchat_lora_finetune.ipynb**: fine-tuning de TinyLlama-1.1B com LoRA para gerar explicações naturais sobre as recomendações deste modelo.
- Os módulos `app/models/vodrec_transformer.py` e `app/services/llm_recommendation_service.py` integram este checkpoint no FastAPI.